In [3]:
# =================================================================
# WEEK 2 DAY 2 - Save Model + Build Prediction API 
# Phetho Tlaka | April 2026
# 
# WHAT WE ARE BUILDING TODAY:
# 
# PROBLEM: Every time Jupyter restarts the trained 
# model dissappears. We have to retrain from scratch.
# In production this is not acceptable
# 
# SOLUTION: Save the model to a file using joblib
# joblib serialieses Python objects to disk 
# Load it back in any script or API instantly.
#
# THEN: Wrap it in a FastAPI web service.
# FASTAPI lets any application see data and 
# get predictions back over HTTP.
# This is how OPenAI, Google and AWS server ML models.
# ====================================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib
import os

print("All libraries loaded!")
print()
print("Today we will:")
print(" 1. Retrain our Titanic model")
print(" 2. Save it to disk joblib")
print(" 3. Load it back and verify it works")
print(" 4. Build a FastAPI prediction endpoint")
print(" 5. Test it live in the browser")

All libraries loaded!

Today we will:
 1. Retrain our Titanic model
 2. Save it to disk joblib
 3. Load it back and verify it works
 4. Build a FastAPI prediction endpoint
 5. Test it live in the browser


In [5]:
# ========================================================================
# STEP 1: RETRAIN THE MODEL
# =========================================================================
# We retrain because Jupyter restarted since Day 1
# This is the exact same code as Day 1 
# In production you would load saved model 
# instead of retraining - that is what we fix today

# Load and clean data 
df = pd.read_csv('../analytics/titanic.csv')
df_clean = df.copy()
df_clean = df_clean.drop(columns=['Cabin'])
df_clean['Age'] = df_clean['Age'].fillna(
                  df_clean['Age'].median())
df_clean['Embarked'] = df_clean['Embarked'].fillna('S')
df_clean['Sex_num'] = df_clean['Sex'].map(
    {'male': 0, 'female': 1})
df_clean['Embarked_num'] = df_clean['Embarked'].map(
    {'S': 0, 'C': 1, 'Q': 2})

# Features and traget 
features = ['Pclass', 'Sex_num', 'Age',
            'SibSp', 'Parch', 'Fare', 'Embarked_num']
x = df_clean[features]
y = df_clean['Survived']

# train/test split 
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42)

# Train model 
model = RandomForestClassifier(
    n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Verify accuracy 
accuracy = accuracy_score(y_test, model.predict(X_test))
print(f"Model retrained!")
print(f"Accuracy: {accuracy * 100:.1f}%")
print(f"Training set: {X_train.shape[0]} passengers")
print(f"Test set:      {X_test.shape[0]} passengers")
print(f"Features:      {features}")

Model retrained!
Accuracy: 82.7%
 Training set: 712 passengers
Test set:      179 passengers
Features:      ['Pclass', 'Sex_num', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked_num']


In [9]:
# =========================================================
# STEP 2: SAVE THE MODEL TO DISK 
# ============================================================
# joblib.dump() serialises the model to a file 
# Think of it like saving a Word document -
# the model is frozen exactly as it is right now
# with all 100 trained decision trees intact 
# 
# We also save the feature list separately
# sow e always know xactly what columns the 
# model expects - in the right order
# 
# MODEL FILE : titanic_model.pkl
# pkl = pickle format - standard for ML models 
# A trained Random Forest is typically 1-5 MB

# Create model directory if it does not exist 
os.makedirs('models', exist_ok=True)

# Save the trained model 
model_path = 'models/titanic_model.pkl'
joblib.dump(model, model_path)
print(f"Model saved to: {model_path}")

# Save the feature list
feature_path = 'models/features.pkl'
joblib.dump(features, feature_path)
print(f"Features saved to: {feature_path}")

# Check file sizes
model_size = os.path.getsize(model_path) / 1024
feature_size = os.path.getsize(feature_path) / 1024
print()
print(f"Model file size:   {model_size:.1f} KB")
print(f"Feature file size: {feature_size:.1f} KB")
print()

# -- VERIFY: Load model back and test it ------
# This proves the saved model works corrrectly
# Load it as if we are a completly fresh script
print("=" * 45)
print("  VERFYING SAVED MODEL")
print("=" * 45)

loaded_model    = joblib.load(model_path)
loaded_features = joblib.load(feature_path)

print(f"Model loaded: {type(loaded_model).__name__}")
print(f"Features:     {loaded_features}")
print()

# Test on the same set
loaded_accuracy = accuracy_score(
    y_test, loaded_model.predict(X_test))
print(f"Loaded model accuracy: "
      f"{loaded_accuracy * 100:.1f}%")
print()

# Quick sanity check - predict one passenger 
test_passenger = pd.DataFrame([{ 
    'Pclass': 1, 'Sex_num': 1, 'Age': 25,
    'SibSp': 0, 'Parch': 0, 'Fare': 100,
    'Embarked_num': 1
}])
prediction = loaded_model.predict(test_passenger)[0]
probability = loaded_model.predict_proba(
    test_passenger)[0][1]

print(f"Sanity check - weathly young woman 1st class:")
print(f" Prediction: "
      f"{'SURVIVED' if prediction == 1 else 'DIED'}")
print(f"  Probability: {probability * 100:.1f}%")
print()
print("=" * 45)
print("  MODEL SAVED AND VERIFIED SUCCESSFULLY")
print("=" * 45)


    


Model saved to: models/titanic_model.pkl
Features saved to: models/features.pkl

Model file size:   2366.6 KB
Feature file size: 0.1 KB

  VERFYING SAVED MODEL
Model loaded: RandomForestClassifier
Features:     ['Pclass', 'Sex_num', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked_num']

Loaded model accuracy: 82.7%

Sanity check - weathly young woman 1st class:
 Prediction: SURVIVED
  Probability: 99.0%

  MODEL SAVED AND VERIFIED SUCCESSFULLY


In [12]:
# ==============================================================
# STEP3: UNDERSTAND WHAT WE ARE BUILDING 
# ==============================================================
# A FastAPI web service - a REST API that:
#
# 1. Runs as a server on your vm 
# 2. Listens for HTTP requests on a port
# 3. Recieves passenger data as JSON
# 4. Loads the saved model 
# 5. returns a prediction JSON 
# 
# HOW IT WORKS IN PRODUCTION:
#
# Client (browser/app)  →  POST /predict  →  API server
#                                              ↓
#                                        loads model
#                                              ↓
#                                        makes prediction
#                                              ↓
# Client  ←  {"survived": 1, "prob": 0.97}  ←  API
# WHY FASTAPI?
# - Fastest Python web framework
# - Auto-generates documentation at /docs
# - Validates incoming data automatically
# - Used by Netflix, Uber and Microsoft
# - One of the most in-demand skills in backend ML 

print("=" * 50)
print(" WHAT WE ARE BUILDING")
print("=" * 50)
print()
print(" A REST API  with these endpoints:")
print()
print("  GET  /       → health check")
print("  GET  /model-info   → model metadata")
print("  POST /predict   → make a prediction")
print("  GET  /docs      → auto API documentation")
print()
print(" Input  (JSON):")
print(" {")
print('   "pclass":   1,')
print('   "sex":      "female",')
print('   "age":      25,')
print('   "sibsp":    0,')
print('   "parch":    0,') 
print('   "fare":     100.0,')
print('   "emabrked": "C",')
print("  }")
print()
print(" Output (JSON):")
print(" {")
print('  "survived":    1,')
print('  "prediction":  "SURVIVED",')
print('  "probability": 0.87,')
print('  "confidence":  "97.9%"')
print(" }")
print()
print(" This is exactly how OpenAI serves GPT-4.")
print(" This is exactly how AWS serves ML models.")
print(" This exactly how Netflix recommends films.")

      

 WHAT WE ARE BUILDING

 A REST API  with these endpoints:

  GET  /       → health check
  GET  /model-info   → model metadata
  POST /predict   → make a prediction
  GET  /docs      → auto API documentation

 Input  (JSON):
 {
   "pclass":   1,
   "sex":      "female",
   "age":      25,
   "sibsp":    0,
   "parch":    0,
   "fare":     100.0,
   "emabrked": "C",
  }

 Output (JSON):
 {
  "survived":    1,
  "prediction":  "SURVIVED",
  "probability": 0.87,
  "confidence":  "97.9%"
 }

 This is exactly how OpenAI serves GPT-4.
 This is exactly how AWS serves ML models.
 This exactly how Netflix recommends films.


In [15]:
# ================================================
# STEP 4: WRITE THE FASTAPI APPLICATION
# ================================================
# We write the API code to a file called main.py
# Then run it from the terminal

api_code = '''
import joblib
import pandas as pd
import numpy as np
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
import uvicorn
import os

# ── Load model once at startup ──────────────────
MODEL_PATH   = "models/titanic_model.pkl"
FEATURE_PATH = "models/features.pkl"

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Model not found at {MODEL_PATH}. "
        "Run the training notebook first."
    )

model    = joblib.load(MODEL_PATH)
features = joblib.load(FEATURE_PATH)

# ── FastAPI app ──────────────────────────────────
app = FastAPI(
    title="Titanic Survival Predictor API",
    description=(
        "Predicts whether a Titanic passenger would "
        "survive based on their details. "
        "Built by Phetho Tlaka | pmtee.github.io"
    ),
    version="1.0.0"
)

# ── Input schema ─────────────────────────────────
# Pydantic validates every incoming request
# If a field is missing or wrong type → 422 error
class Passenger(BaseModel):
    pclass:   int   = Field(..., ge=1, le=3,
                      description="Ticket class 1, 2 or 3")
    sex:      str   = Field(...,
                      description="male or female")
    age:      float = Field(..., ge=0, le=120,
                      description="Age in years")
    sibsp:    int   = Field(0, ge=0,
                      description="Siblings/spouses aboard")
    parch:    int   = Field(0, ge=0,
                      description="Parents/children aboard")
    fare:     float = Field(..., ge=0,
                      description="Ticket price in pounds")
    embarked: str   = Field("S",
                      description="Port: S, C or Q")

# ── Encode helpers ───────────────────────────────
def encode_sex(sex: str) -> int:
    s = sex.strip().lower()
    if s not in ("male", "female"):
        raise HTTPException(
            status_code=422,
            detail=f"sex must be male or female, got: {sex}"
        )
    return 1 if s == "female" else 0

def encode_embarked(port: str) -> int:
    p = port.strip().upper()
    mapping = {"S": 0, "C": 1, "Q": 2}
    if p not in mapping:
        raise HTTPException(
            status_code=422,
            detail=f"embarked must be S, C or Q, got: {port}"
        )
    return mapping[p]

# ── Routes ───────────────────────────────────────
@app.get("/")
def health_check():
    return {
        "status":  "healthy",
        "model":   "RandomForestClassifier",
        "accuracy":"82.7%",
        "author":  "Phetho Tlaka",
        "docs":    "/docs"
    }

@app.get("/model-info")
def model_info():
    return {
        "model_type":    type(model).__name__,
        "n_estimators":  model.n_estimators,
        "features":      features,
        "n_features":    len(features),
        "accuracy":      "82.7%",
        "trained_on":    "891 Titanic passengers",
        "test_set_size": "179 passengers"
    }

@app.post("/predict")
def predict(passenger: Passenger):
    # Encode text fields to numbers
    sex_num      = encode_sex(passenger.sex)
    embarked_num = encode_embarked(passenger.embarked)

    # Build feature DataFrame in correct order
    data = pd.DataFrame([{
        "Pclass":       passenger.pclass,
        "Sex_num":      sex_num,
        "Age":          passenger.age,
        "SibSp":        passenger.sibsp,
        "Parch":        passenger.parch,
        "Fare":         passenger.fare,
        "Embarked_num": embarked_num
    }])

    # Make prediction
    prediction  = int(model.predict(data)[0])
    probability = float(
        model.predict_proba(data)[0][1])

    return {
        "survived":    prediction,
        "prediction":  "SURVIVED" if prediction == 1
                       else "DIED",
        "probability": round(probability, 4),
        "confidence":  f"{probability * 100:.1f}%",
        "input": {
            "pclass":   passenger.pclass,
            "sex":      passenger.sex,
            "age":      passenger.age,
            "fare":     passenger.fare,
            "embarked": passenger.embarked
        }
    }

# ── Run ──────────────────────────────────────────
if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

# Write the API file
with open('main.py', 'w') as f:
    f.write(api_code)

print("main.py written successfully!")
print()
print("File contents:")
print(f"  Location: ml/main.py")
print(f"  Size:     {len(api_code)} characters")
print()
print("Next steps:")
print("  1. Open a NEW terminal in Jupyter")
print("     File → New → Terminal")
print("  2. Run:")
print("     cd ~/projects/phetho-lab/ml")
print("     pip install fastapi uvicorn --quiet")
print("     python main.py")
print("  3. Open browser and go to:")
print("     http://127.0.0.1:8000/docs")

main.py written successfully!

File contents:
  Location: ml/main.py
  Size:     4187 characters

Next steps:
  1. Open a NEW terminal in Jupyter
     File → New → Terminal
  2. Run:
     cd ~/projects/phetho-lab/ml
     pip install fastapi uvicorn --quiet
     python main.py
  3. Open browser and go to:
     http://127.0.0.1:8000/docs
